In [0]:
# Databricks notebook source
# MAGIC %md # Deploy anomaly_detector2 v2 (PyFunc Wrapper for UC Serving)

# COMMAND ----------
# MAGIC %md ## Step 0 — Install / confirm dependencies
# COMMAND ----------
# MAGIC %pip install mlflow scikit-learn pandas --quiet
# MAGIC dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md ## Step 1 — Imports & Configuration
# COMMAND ----------
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from mlflow import MlflowClient
import pandas as pd
import pickle
import logging

# Ensure these files are accessible in your workspace/repo
from feature_engineering import ALL_FEATURES
from train_model import IntegratedAnomalyDetector

CATALOG         = "capstone_project"
SCHEMA          = "capstone_schema"
MODEL_NAME      = "anomaly_detector2"
METRICS_TABLE   = f"{CATALOG}.{SCHEMA}.pipeline_monitoring_metrics"
UC_MODEL_PATH   = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
EXPERIMENT_NAME = f"/Users/capstoneproject196@gmail.com/{MODEL_NAME}_training"   # Update with your username

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

# COMMAND ----------
# MAGIC %md ## Step 2 — Define the PyFunc Wrapper Class
# MAGIC This intercepts the DataFrame sent by UC Serving, extracts the row as a dict, 
# MAGIC and passes it to your custom detector class.
# COMMAND ----------
class AnomalyDetectorWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with open(context.artifacts["detector_model"], "rb") as f:
            self.detector = pickle.load(f)

    def predict(self, context, model_input):
        # Model Serving passes a DataFrame. Your class expects a dict.
        if isinstance(model_input, pd.DataFrame):
            input_dict = model_input.iloc[0].to_dict()
        else:
            input_dict = model_input
            
        return self.detector.predict(input_dict)

# COMMAND ----------
# MAGIC %md ## Step 3 — Load Data & Train the Model
# COMMAND ----------
logger.info(f"Loading data from {METRICS_TABLE} ...")
df = spark.table(METRICS_TABLE).toPandas()

if len(df) < 15:
    raise ValueError(f"Not enough data in {METRICS_TABLE}. Run telemetry generation first.")

detector = IntegratedAnomalyDetector(contamination=0.05, temp=3.0)
detector.train(df)

# Save the trained underlying scikit-learn model temporarily
model_path = "/tmp/detector.pkl"
with open(model_path, "wb") as f:
    pickle.dump(detector, f)

# COMMAND ----------
# MAGIC %md ## Step 4 — Define Signature and Log to Unity Catalog
# COMMAND ----------
# Build schemas (Inputs: Doubles, Outputs: Mixed types mapping to the predict dict)
input_schema = Schema([ColSpec("double", feat) for feat in ALL_FEATURES])
output_schema = Schema([
    ColSpec("string", "Anomaly_Detected"),
    ColSpec("double", "Severity_Score"),
    ColSpec("double", "Volume_Spike_Probability"),
    ColSpec("double", "Schema_Quality_Probability"),
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# ... (Previous code remains the same) ...

# COMMAND ----------
# MAGIC %md ## Step 4 — Define Signature and Log to Unity Catalog
# COMMAND ----------

# ... (Schema definition remains the same) ...

with mlflow.start_run(run_name="anomaly_detector2_v2_wrapper") as run:
    
    # Log hyperparameters
    mlflow.log_param("contamination", detector.contamination)
    mlflow.log_param("temperature", detector.temp)
    mlflow.log_param("n_features", len(ALL_FEATURES))
    
    # Log the PyFunc model 
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=AnomalyDetectorWrapper(),
        artifacts={"detector_model": model_path},
        code_paths=["train_model.py", "feature_engineering.py"], # 👈 THE FIX IS HERE
        registered_model_name=UC_MODEL_PATH,
        signature=signature
    )
    
    # Extract the version number directly from the logging response
    NEW_VERSION = model_info.registered_model_version
    logger.info(f"Successfully logged Version {NEW_VERSION} to {UC_MODEL_PATH}")

# COMMAND ----------
# MAGIC %md ## Step 5 — Move @champion Alias to the New Version
# COMMAND ----------
client = MlflowClient()

client.set_registered_model_alias(
    name=UC_MODEL_PATH,
    alias="champion",
    version=NEW_VERSION
)

print(f"Alias @champion successfully moved to version {NEW_VERSION}")

# COMMAND ----------
# MAGIC %md ## Step 6 — PyFunc Smoke Test (Using DataFrame)
# COMMAND ----------
loaded_pyfunc = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_PATH}@champion")

# UC Serving sends inputs as a DataFrame, so we test with one.
test_df = pd.DataFrame([{
    "current_volume": 5000.0, "pct_change": 0.01, 
    "lag_1h": 4990.0, "lag_2h": 4980.0, "lag_3h": 4970.0, 
    "rolling_mean_3h": 4980.0, "rolling_mean_6h": 4980.0, "rolling_std_6h": 5.0, 
    "hour_sin": 0.5, "hour_cos": 0.8, "is_weekend": 0.0, 
    "null_pct": 0.0, "dup_pct": 0.0, 
    "mismatch_count": 0.0, "missingcol_count": 0.0
}])

prediction = loaded_pyfunc.predict(test_df)

print("\nSmoke-test prediction (normal traffic input):")
for k, v in prediction.items():
    print(f"  {k}: {v}")

assert prediction["Anomaly_Detected"] == "NO", "Smoke test FAILED — expected NO anomaly for normal input."
print("\nSmoke test PASSED. Model is ready for serving.")